In [ ]:
!pip install mediapipe gradio
import mediapipe as mp
import cv2
import numpy as np
import gradio as gr

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [ ]:
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [ ]:
def check_pushup_feedback(landmarks):
    left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
    left_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
    left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

    right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
    right_elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
    right_wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

    left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
    right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
    avg_elbow_angle = (left_elbow_angle + right_elbow_angle) / 2

    left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]

    body_line_angle = calculate_angle(left_shoulder, left_hip, left_knee)

    if avg_elbow_angle < 80:
        feedback = "Too Deep! Don't Strain Elbows"
    elif 80 <= avg_elbow_angle <= 100:
        feedback = "Perfect Depth!"
    elif 100 < avg_elbow_angle <= 165:
        feedback = "Almost There! Go Lower"
    else:
        feedback = "Too High! Bend Your Elbows Deeper"

    if not (body_line_angle >= 165 and body_line_angle <= 195):
        if body_line_angle < 165:
            feedback = "Hips Too Low! Keep Body Straight"
        elif body_line_angle > 195:
            feedback = "Hips Too High! Keep Body Straight"

    depth_score = max(0, 100 - abs(90 - avg_elbow_angle) * 1.5)
    body_score = max(0, 100 - abs(180 - body_line_angle) * 3)

    accuracy = (depth_score * 0.4) + (body_score * 0.6)
    accuracy = max(0, min(100, int(accuracy)))

    return feedback, int(accuracy)

In [ ]:
def analyze_pushups(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))

    # Fix for video speed (stable FPS)
    fps = 30

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    output_video = "output_pushup.mp4"
    out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Fix for mirrored video (un-mirroring webcam view)
        frame = cv2.flip(frame, 1)

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            landmarks = results.pose_landmarks.landmark

            feedback, accuracy = check_pushup_feedback(landmarks)

            color = (0, 255, 0) if "Perfect" in feedback else (0, 0, 255)

            cv2.putText(image, feedback, (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)
            cv2.putText(image, f"Accuracy: {accuracy}%", (50, 100), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)

        out.write(image)

    cap.release()
    out.release()
    return output_video

gr.Interface(
    fn=analyze_pushups,
    inputs=gr.Video(sources=["upload", "webcam"]),
    outputs=gr.Video(),
    title="Push-Up Form Analyzer",
    description="Upload a video of your push-ups, and get feedback on your form!",
).launch(share=True, debug=True)